# 06: Scoreline probabilities and evaluation

Turn the two expected-goals numbers from notebook 05 into a probability for every possible scoreline, add those up into home win, draw and away win probabilities, and judge them fairly against simple baselines and the bookmakers on the 2024/25 validation season. The 2025/26 test season is not used here.

**Headline result (375 validation matches):** the Poisson model reaches a log loss of 1.019, against 1.078 for the constant baseline and 0.971 for the bookmakers, and picks the right result 49.6% of the time (41.1% for always predicting a home win, 54.1% for the bookmakers).

**Contents**
1. Setup: refit the models from notebook 05
2. The scoreline grid
3. From scorelines to home win / draw / away win
4. Fair comparison with baselines and bookmakers
5. Exact scoreline hit rate
6. Calibration
7. Predicting a single past match (`check_match`)
8. Summary and next steps

## 1. Setup: refit the models from notebook 05

Repeat the loading, the split and the Poisson fit from notebook 05 so that this notebook runs on its own. It produces `lam_home` and `lam_away`: the expected home goals and expected away goals (written lambda) for every validation match, from models trained on the seasons up to 2023/24 only.

In [22]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss

df = pd.read_csv("../data/processed/model_table.csv",
                 parse_dates=["Date"], dtype={"Season": str})
df["h_rest"] = df["h_rest"].clip(upper=14)
df["a_rest"] = df["a_rest"].clip(upper=14)

features = [
    "h_points_last5", "h_gf5", "h_ga5", "h_gf10", "h_ga10", "h_rest",
    "a_points_last5", "a_gf5", "a_ga5", "a_gf10", "a_ga10", "a_rest",
]
train = df[df["Season"] <= "2324"]
valid = df[df["Season"] == "2425"]

def fit_poisson(target):
    model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))
    model.fit(train[features], train[target])
    return model

model_home = fit_poisson("home_goals")
model_away = fit_poisson("away_goals")
lam_home = model_home.predict(valid[features])
lam_away = model_away.predict(valid[features])

## 2. The scoreline grid

If a team is expected to score lambda goals, the Poisson distribution gives the probability that it scores exactly 0, 1, 2, 3... goals. Assuming the two teams score independently, the probability of a scoreline is the product of the two: P(2-1) = P(home scores 2) x P(away scores 1).

The function builds, for every match, an 11 x 11 table: rows are home goals (0 to 10), columns are away goals (0 to 10), and each cell is the probability of that exact score. It uses NumPy broadcasting to handle all 375 matches at once.

In [23]:
MAX_GOALS = 10
goals = np.arange(MAX_GOALS + 1)

def scoreline_grid(lam_h, lam_a):
    """Array of shape (matches, home goals, away goals) with scoreline probabilities."""
    p_h = poisson.pmf(goals[None, :], np.asarray(lam_h)[:, None])   # (n, 11)
    p_a = poisson.pmf(goals[None, :], np.asarray(lam_a)[:, None])   # (n, 11)
    return p_h[:, :, None] * p_a[:, None, :]                        # (n, 11, 11)

grid = scoreline_grid(lam_home, lam_away)
grid.shape, grid.sum(axis=(1, 2)).min()

((375, 11, 11), np.float64(0.999891162147293))

**Result:** the grid has shape (375, 11, 11) and the smallest total probability of any match is 0.99989, so at least 99.989% of the probability lies inside the grid in every match. Nothing important is lost by stopping at 10 goals.

## 3. From scorelines to home win / draw / away win

Mark which cells of the grid mean a home win (home goals > away goals), a draw (equal) and an away win, then add up the probability in each group. Dividing by the grid total renormalises away the tiny tail beyond 10 goals.

In [24]:
I, J = np.meshgrid(goals, goals, indexing="ij")    # I = home goals, J = away goals

def result_probs(lam_h, lam_a):
    g = scoreline_grid(lam_h, lam_a)
    t = g.sum(axis=(1, 2))
    return np.column_stack([(g * (I > J)).sum(axis=(1, 2)) / t,     # home win
                            (g * (I == J)).sum(axis=(1, 2)) / t,    # draw
                            (g * (I < J)).sum(axis=(1, 2)) / t])    # away win

probs = pd.DataFrame(result_probs(lam_home, lam_away),
                     columns=["H", "D", "A"], index=valid.index)
probs.sum(axis=1).round(6).unique()      # should be [1.]

array([1.])

**Result:** the three probabilities add up to 1 for every match.

### One match in detail

Look at the first validation match.

In [25]:
k = 0
print(valid.iloc[k][["HomeTeam", "AwayTeam", "home_goals", "away_goals"]].to_dict())
print(f"expected goals: {lam_home[k]:.2f} - {lam_away[k]:.2f}")
print(probs.iloc[k].round(3).to_dict())
best = np.unravel_index(grid[k].argmax(), grid[k].shape)
print("most likely scoreline:", best, "with probability", round(grid[k][best], 3))

{'HomeTeam': 'Man United', 'AwayTeam': 'Fulham', 'home_goals': 1, 'away_goals': 0}
expected goals: 1.65 - 1.25
{'H': 0.467, 'D': 0.242, 'A': 0.29}
most likely scoreline: (np.int64(1), np.int64(1)) with probability 0.114


**Result:** for Man United vs Fulham (16 Aug 2024, actual score 1-0) the model expects 1.65 home goals and 1.25 away goals, and gives 46.7% to a home win, 24.2% to a draw and 29.0% to an away win. The most likely single scoreline is 1-1 at only 11.4%, even though a home win is the most likely result. Many different scorelines add up to a home win, while each individual score is a small slice, which is why probabilities are more informative than one guessed score.

## 4. Fair comparison with baselines and bookmakers

Every contender is scored on the same 375 validation matches (all of which have bookmaker odds), with two metrics:
- **Accuracy:** how often the most probable result was the real one. Simple, but it ignores how confident the model was.
- **Log loss:** for each match, take the probability given to what really happened and compute minus its natural log. Giving 60% to the real result costs 0.51 and giving 10% costs 2.30, so confident mistakes are punished hard. Lower is better, and guessing 1/3 for every result scores 1.099.

Log loss is computed by hand in `metrics`. scikit-learn's `log_loss` assumes the probability columns are in alphabetical order (A, D, H), while these are in the order H, D, A, so using it directly would silently swap home and away.

In [26]:
# Only matches with bookmaker odds, so every model is judged on the same rows
mask = valid[["B365H", "B365D", "B365A"]].notna().all(axis=1)
v = valid[mask]
y = v["FTR"].to_numpy()
labels = ["H", "D", "A"]

def metrics(name, p):
    p = np.asarray(p)
    pred = np.array(labels)[p.argmax(axis=1)]
    true_idx = pd.Series(y).map({"H": 0, "D": 1, "A": 2}).to_numpy()
    prob_of_truth = np.clip(p[np.arange(len(p)), true_idx], 1e-12, None)
    return {"model": name,
            "accuracy": (pred == y).mean(),
            "log loss": -np.log(prob_of_truth).mean()}

**The four contenders**
1. **Constant:** the training seasons' result shares (44.8% home, 22.5% draw, 32.7% away) for every match. Its accuracy is the accuracy of "always predict a home win".
2. **Baseline goals to Poisson:** the same average goals for every match, run through the same grid. It knows about home advantage but nothing about the teams.
3. **Poisson regression:** the model's expected goals run through the grid.
4. **Bookmaker (Bet365):** odds of 2.00 imply 50%, but a bookmaker's three implied probabilities add up to more than 100%. The excess is the margin, and dividing by the total removes it.

In [27]:
# 1. Constant: the training result shares for every match (its accuracy = "always home win")
shares = train["FTR"].value_counts(normalize=True)[labels]
p_const = np.tile(shares.to_numpy(), (len(v), 1))

# 2. Baseline goals: the same average goals for every match, pushed through the same grid
p_base = result_probs(np.full(len(v), train["home_goals"].mean()),
                      np.full(len(v), train["away_goals"].mean()))

# 3. Your model
p_model = result_probs(lam_home[mask.to_numpy()], lam_away[mask.to_numpy()])

# 4. Bookmaker: invert the odds, then remove the margin by normalising
raw = 1 / v[["B365H", "B365D", "B365A"]].to_numpy()
p_book = raw / raw.sum(axis=1, keepdims=True)

table = pd.DataFrame([
    metrics("Constant (training result shares)", p_const),
    metrics("Baseline goals -> Poisson", p_base),
    metrics("Poisson regression", p_model),
    metrics("Bookmaker odds (Bet365)", p_book),
]).round(3)
table

,model,accuracy,log loss
0,Constant (training result shares),0.411,1.078
1,Baseline goals -> Poisson,0.411,1.078
2,Poisson regression,0.496,1.019
3,Bookmaker odds (Bet365),0.541,0.971


| Model | Accuracy | Log loss |
|---|---|---|
| Constant (training result shares) | 41.1% | 1.078 |
| Baseline goals to Poisson | 41.1% | 1.078 |
| Poisson regression | 49.6% | 1.019 |
| Bookmaker odds (Bet365) | 54.1% | 0.971 |

- **The model clearly beats the baselines.** Log loss falls from 1.078 to 1.019 (5.5% lower) and accuracy rises by 8.5 percentage points.
- **The two baselines tie** because both ignore the teams and give almost the same probabilities (about 44/25/31 against 45/23/33), so their scores agree to three decimals.
- **The bookmakers are still better** (0.971 and 54.1%), as expected: their odds reflect lineups, injuries and betting-market information that the model does not have. The model closes roughly 55% of the log-loss gap between the baseline and the bookmakers.
- **Do not over-read accuracy.** With 375 matches, an accuracy figure carries about 2.6 percentage points of sampling noise (one standard error), so the 4.5-point gap to the bookmakers is under two standard errors. Log loss is the steadier comparison.
- Neither the model nor the bookmakers picked a draw as the most likely result in any match, which caps the accuracy of any approach that picks one result per match.

## 5. Exact scoreline hit rate

Find the biggest cell in each grid (the single most likely scoreline) and check how often that exact score really happened.

In [28]:
best_idx = grid.reshape(len(grid), -1).argmax(axis=1)
pred_h, pred_a = np.divmod(best_idx, MAX_GOALS + 1)
exact = ((pred_h == valid["home_goals"].to_numpy()) &
         (pred_a == valid["away_goals"].to_numpy())).mean()
print(f"Exact scoreline hit rate: {exact:.1%}")
pd.Series(list(zip(pred_h, pred_a))).value_counts().head()

Exact scoreline hit rate: 12.3%


(1, 1)    305
(2, 0)     31
(1, 0)     19
(2, 1)     10
(0, 2)      7
Name: count, dtype: int64

**Result:** 12.3% (46 of 375 matches). This is not evidence of skill. The top cell is 1-1 in 305 of the 375 matches, because with expected goals around 1.6 and 1.3 the single most likely score is almost always 1-1, and 1-1 is also the most common real scoreline (43 of 375 matches, 11.5%). A "model" that always said 1-1 would score about the same. Exact scores are inherently hard to predict, and probabilities are the better output.

## 6. Calibration

Compare the average predicted probability of each result with how often it really happened. A well-calibrated model's averages match reality.

In [29]:
calib = pd.DataFrame({
    "predicted share": probs.mean(),
    "actual share": valid["FTR"].value_counts(normalize=True)[labels],
}).round(3)
calib

,predicted share,actual share
H,0.442,0.411
D,0.236,0.240
A,0.322,0.349


| | Predicted | Actual |
|---|---|---|
| Home win | 44.2% | 41.1% |
| Draw | 23.6% | 24.0% |
| Away win | 32.2% | 34.9% |

Draws are almost perfectly calibrated, so independent Poisson goals do not under-predict draws here. The model gives home wins about 3 points too much and away wins about 3 points too little. The validation season had fewer home wins (41.1%) than the training seasons (44.8%), a shift that a model trained on earlier seasons could not know about.

## 7. Predicting a single past match

`check_match(home, away, season="2425", my_guess=None)` takes a match that has already been played and reports:
- the expected goals and the model's home win / draw / away win probabilities, next to the bookmaker's (margin removed)
- the five most likely scorelines
- the model's predicted score: the most likely scoreline **within the result the model picked**. The overall top cell is almost always 1-1, so this avoids contradictions such as "predicts a home win, score 1-1"
- an optional guess of your own, so that you can compare your call with the model's
- the actual result, whether each prediction got the exact score and the result right, and the probability the model gave to what really happened

The features come from `df`, which only uses matches before kickoff, and the models were trained on seasons up to 2023/24, so a 2024/25 match is a fair test. Team names must match the data (`Man City`, not `Manchester City`). A wrong name raises an error that lists the valid ones.

In [35]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

def check_match(home, away, season="2425", my_guess=None, max_goals=10):
    """Predict a match that has already been played and compare it with the real result."""
    rows = df[(df["Season"] == season) & (df["HomeTeam"] == home) & (df["AwayTeam"] == away)]
    if rows.empty:
        teams = sorted(df.loc[df["Season"] == season, "HomeTeam"].unique())
        raise ValueError(f"No {home} vs {away} match found in {season}. Team names: {teams}")
    r = rows.iloc[0]

    def outcome(h, a):
        return "H" if h > a else "A" if h < a else "D"

    # 1. Expected goals from the two Poisson models (features use only earlier matches)
    X = rows[features].iloc[[0]]
    lam_h = model_home.predict(X)[0]
    lam_a = model_away.predict(X)[0]

    # 2. Scoreline grid: rows = home goals, columns = away goals
    goals = np.arange(max_goals + 1)
    grid = np.outer(poisson.pmf(goals, lam_h), poisson.pmf(goals, lam_a))
    grid = grid / grid.sum()

    # 3. Result probabilities
    I, J = np.indices(grid.shape)
    masks = {"H": I > J, "D": I == J, "A": I < J}
    probs = {k: grid[m].sum() for k, m in masks.items()}
    names = {"H": f"{home} win", "D": "Draw", "A": f"{away} win"}
    predicted = max(probs, key=probs.get)

    # 4. Predicted score: most likely scoreline within the predicted result
    sub = np.where(masks[predicted], grid, 0)
    ph, pa = np.unravel_index(sub.argmax(), grid.shape)

    # 5. Bookmaker probabilities with the margin removed (if odds exist)
    odds = r[["B365H", "B365D", "B365A"]].to_numpy(dtype=float)
    book = None
    if not np.isnan(odds).any():
        raw = 1 / odds
        book = dict(zip(["H", "D", "A"], raw / raw.sum()))

    # 6. Actual result from the data
    hg, ag, actual = int(r["home_goals"]), int(r["away_goals"]), r["FTR"]

    # 7. Report
    top = np.argsort(grid.ravel())[::-1][:5]
    top_scores = [divmod(k, max_goals + 1) for k in top]

    print(f"{home} vs {away}  |  {r['Date'].date()}  |  season {season}")
    print("-" * 64)
    print(f"Expected goals:  {home} {lam_h:.2f}  -  {lam_a:.2f} {away}")
    print("Model:      " + " | ".join(f"{names[k]} {probs[k]:.1%}" for k in "HDA"))
    if book:
        print("Bookmaker:  " + " | ".join(f"{names[k]} {book[k]:.1%}" for k in "HDA"))
    print("Most likely scorelines: " + ", ".join(f"{i}-{j} ({grid[i, j]:.1%})" for i, j in top_scores))
    print("-" * 64)
    print(f"MODEL'S PREDICTED SCORE:  {home} {ph}-{pa} {away}  ({names[predicted]}, scoreline probability {grid[ph, pa]:.1%})")
    if my_guess is not None:
        gh, ga = my_guess
        print(f"YOUR GUESS:               {home} {gh}-{ga} {away}  ({names[outcome(gh, ga)]}, model probability {grid[gh, ga]:.1%})")
    print("-" * 64)
    print(f"ACTUAL:  {home} {hg}-{ag} {away}  ->  {names[actual]}")

    def verdict(label, h, a):
        exact = "yes" if (h, a) == (hg, ag) else "no"
        res = "yes" if outcome(h, a) == actual else "no"
        err = abs(h - hg) + abs(a - ag)
        print(f"{label:<6} exact score: {exact:<3} | correct result: {res:<3} | total goal error: {err}")

    verdict("Model", ph, pa)
    if my_guess is not None:
        verdict("You", gh, ga)
    print(f"Probability the model gave to the actual result: {probs[actual]:.1%}")
    if hg <= max_goals and ag <= max_goals:
        print(f"Probability the model gave to the exact score {hg}-{ag}: {grid[hg, ag]:.1%}")

check_match("Chelsea", "Arsenal", my_guess=(1, 3))

Chelsea vs Arsenal  |  2024-11-10  |  season 2425
----------------------------------------------------------------
Expected goals:  Chelsea 1.66  -  1.25 Arsenal
Model:      Chelsea win 47.0% | Draw 24.2% | Arsenal win 28.8%
Bookmaker:  Chelsea win 35.1% | Draw 27.8% | Arsenal win 37.1%
Most likely scorelines: 1-1 (11.3%), 2-1 (9.4%), 1-0 (9.1%), 2-0 (7.5%), 1-2 (7.1%)
----------------------------------------------------------------
MODEL'S PREDICTED SCORE:  Chelsea 2-1 Arsenal  (Chelsea win, scoreline probability 9.4%)
YOUR GUESS:               Chelsea 1-3 Arsenal  (Arsenal win, model probability 2.9%)
----------------------------------------------------------------
ACTUAL:  Chelsea 1-1 Arsenal  ->  Draw
Model  exact score: no  | correct result: no  | total goal error: 1
You    exact score: no  | correct result: no  | total goal error: 2
Probability the model gave to the actual result: 24.2%
Probability the model gave to the exact score 1-1: 11.3%


**Example run:** Chelsea vs Arsenal on 10 Nov 2024, with a guess of 1-3.
- The model expected 1.66 Chelsea goals and 1.25 Arsenal goals and gave Chelsea 47.0%, a draw 24.2% and Arsenal 28.8%. The bookmakers were more cautious: Chelsea 35.1%, a draw 27.8% and Arsenal 37.1%. The model's predicted score was Chelsea 2-1 Arsenal.
- The match finished 1-1. The model gave the actual result 24.2% and the exact score 11.3%, and the bookmakers gave the result 27.8%.
- A single match proves nothing, since a 47% favourite fails to win more often than not. It does show the model's blind spot: it only sees recent form, so it rated Chelsea far above what the bookmakers did, and their odds reflect more information, such as overall team strength.

## 8. Summary and next steps

**What was done**
- Turned each match's expected home and away goals into a probability for every scoreline (0-0 to 10-10), then into home win, draw and away win probabilities
- Compared four approaches on the same 375 validation matches with accuracy and log loss
- Checked the exact-score hit rate and the calibration of the probabilities
- Built `check_match` to predict any past match and compare it with the real result

**Results on the validation season (2024/25, 375 matches)**

| Model | Accuracy | Log loss |
|---|---|---|
| Constant (training result shares) | 41.1% | 1.078 |
| Baseline goals to Poisson | 41.1% | 1.078 |
| Poisson regression | 49.6% | 1.019 |
| Bookmaker odds (Bet365) | 54.1% | 0.971 |

**Main findings**
- The Poisson model beats both baselines and closes roughly 55% of the log-loss gap to the bookmakers, using only recent-form features.
- It does not beat the bookmakers, who use information the model lacks.
- The 12.3% exact-score hit rate mostly reflects that 1-1 is common, so it is not a measure of skill.
- Draw probabilities are well calibrated. Home wins are over-predicted by about 3 points in this season.

**Limitations**
- One validation season of 375 matches, so differences of a few points are noisy
- Recent form only: no team-strength rating, lineups or injuries
- Home and away goals are assumed independent, which slightly misjudges low scores and draws
- A model trained on earlier seasons cannot know that 2024/25 had fewer home wins

**Next steps**
1. Add a team-strength feature (for example an Elo rating), a natural next step because recent form alone cannot separate strong teams from weak ones
2. Optionally try LightGBM with a Poisson objective as a second model
3. Choose the final model on the validation season, then evaluate it once on the 2025/26 test season
4. Refit on all completed seasons, save the models, and build the Streamlit demo